# experiment:olmoearth_lora_unet_496
## OlmoEarth + LoRA Encoder + UNet 496 Decoder

- Phase 1 (5 ep): decoder only, LoRA frozen.
- Phase 2 (95 ep): decoder + LoRA jointly.
- No pre-computed embeddings — live forward pass every batch.

## 0. Config

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

DATASET_DIR         = '/content/drive/MyDrive/CS682/project/RiverScope_dataset'
OUTPUT_DIR          = '/content/drive/MyDrive/CS682/project/results/olmoearth_lora_unet_496'
RIVERSCOPE_REPO_DIR = '/content/drive/MyDrive/CS682/project/riverscope-models'
UNET496_BEST_CKPT   = '/content/drive/MyDrive/CS682/project/results/olmoearth_unet_496_v2_LATEST/checkpoints/best_5e-04.pt'
MODEL_SIZE     = 'OLMOEARTH_V1_BASE'
PATCH_SIZE     = 8
INPUT_SIZE     = 496
NUM_CLASSES    = 2
PHASE1_EPOCHS  = 5
PHASE2_EPOCHS  = 95
BATCH_SIZE     = 16
RANDOM_SEED    = 42
DICE_WEIGHT    = 0.5
LORA_RANK      = 8
LORA_ALPHA     = 16.0
LR_DECODER     = 5e-4
LR_LORA        = 1e-4
MASK_KEY       = 'masks_448'

## 1. Environment Setup

In [ ]:
import sys, os
# ── Add RiverDecode to path so `riverdecode` is importable ──────────────────
RIVERDECODE_DIR = '/content/drive/MyDrive/CS682/project/riverscope-models/RiverDecode'
if RIVERDECODE_DIR not in sys.path:
    sys.path.insert(0, RIVERDECODE_DIR)

from riverdecode.setup import (
    mount_drive, patch_dynamo, setup_output_dirs, print_gpu_info,
    install_dependencies, clone_or_update_repo,
)
from riverdecode.data      import set_seeds, load_splits, parse_all_splits, load_resolution_ceiling, print_date_stats
from riverdecode.io        import read_image_for_model, read_image_for_display, read_mask_raw, verify_sample_tile
from riverdecode.backbone  import load_olmoearth, extract_spatial_tokens
from riverdecode.embeddings import load_or_compute_embeddings
from riverdecode.losses    import (
    combined_loss_bce, combined_loss_ce,
    compute_pos_weight, compute_f1_sigmoid, compute_f1_argmax,
    compute_iou_np, compute_f1_np, build_ce_criterion,
)
from riverdecode.training  import adjust_learning_rate, train_one_lr, SweepState, val_sanity_viz

In [ ]:
mount_drive()
install_dependencies()
clone_or_update_repo(RIVERSCOPE_REPO_DIR)
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print_gpu_info()
setup_output_dirs(OUTPUT_DIR)
patch_dynamo()

## 2. Dataset

In [ ]:
set_seeds(RANDOM_SEED)
df_train, df_valid, df_test = load_splits(DATASET_DIR)
df_train, df_valid, df_test = parse_all_splits(df_train, df_valid, df_test)
print_date_stats(df_train, df_valid, df_test)

## 3. Load Encoder + Inject LoRA

In [ ]:
from riverdecode.models.lora_unet     import inject_lora
from riverdecode.models.unet_extended import OlmoEarthUNetDecoderExtended

olmo_model, EMBED_DIM = load_olmoearth(MODEL_SIZE, DEVICE)
n_lora = inject_lora(olmo_model, rank=LORA_RANK, alpha=LORA_ALPHA)
print(f'LoRA adapters injected: {n_lora}')

## 4. Decoder + Load 496 Checkpoint

In [ ]:
decoder = OlmoEarthUNetDecoderExtended(embed_dim=EMBED_DIM, num_classes=NUM_CLASSES).to(DEVICE)

if os.path.exists(UNET496_BEST_CKPT):
    ckpt = torch.load(UNET496_BEST_CKPT, map_location=DEVICE)
    state = ckpt.get('model_state_dict', ckpt)
    missing, unexpected = decoder.load_state_dict(state, strict=False)
    print(f'Loaded UNet-496 checkpoint — missing: {len(missing)}, unexpected: {len(unexpected)}')
    # Reset head to avoid bias from prior training resolution
    import torch.nn as nn
    decoder.head = nn.Conv2d(64, NUM_CLASSES, kernel_size=1).to(DEVICE)
    nn.init.kaiming_normal_(decoder.head.weight, mode='fan_out', nonlinearity='relu')
    nn.init.zeros_(decoder.head.bias)
    print('Head reset to random init.')
else:
    print(f'WARNING: checkpoint not found at {UNET496_BEST_CKPT} — training from scratch.')

## 5. pos_weight from Existing Embeddings

In [ ]:
_emb_path = os.path.join(
    os.path.dirname(OUTPUT_DIR),
    'olmoearth_unet_496_v2_LATEST', 'embeddings', 'train_embeddings.pt'
)
POS_WEIGHT = compute_pos_weight(_emb_path, mask_key=MASK_KEY)
pos_weight_tensor = torch.tensor(POS_WEIGHT, device=DEVICE)

## 6. Dataset Builders (Live Forward Pass — No Cached Embeddings)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class LiveDataset(Dataset):
    def __init__(self, df, dataset_dir):
        self.df, self.dataset_dir = df.reset_index(drop=True), dataset_dir
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img_path   = os.path.join(self.dataset_dir, row['normalized_planetscope_path'])
        label_path = os.path.join(self.dataset_dir, row['label_path'])
        img  = read_image_for_model(img_path)
        mask = read_mask_raw(label_path)
        import torch, torch.nn.functional as F
        img_t  = torch.from_numpy(img).unsqueeze(0)
        img_t  = F.interpolate(img_t, size=(INPUT_SIZE, INPUT_SIZE), mode='bilinear', align_corners=False).squeeze(0)
        mask_t = torch.from_numpy(mask).unsqueeze(0).unsqueeze(0)
        mask_t = F.interpolate(mask_t, size=(448, 448), mode='nearest').squeeze().long()
        return img_t, mask_t, int(row['_day']), int(row['_month']), int(row['_year'])

train_ds = LiveDataset(df_train, DATASET_DIR)
valid_ds = LiveDataset(df_valid, DATASET_DIR)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

## 7. Loss & Metric

In [ ]:
import torch.nn as nn
criterion = build_ce_criterion(POS_WEIGHT, DEVICE)

def loss_fn_live(logits, masks):
    return combined_loss_ce(logits, masks, criterion, dice_weight=DICE_WEIGHT)

## 8. Phase 1 — Decoder Only (LoRA frozen)

In [ ]:
from tqdm.auto import tqdm

# Freeze LoRA during Phase 1
for name, p in olmo_model.named_parameters():
    if 'lora' in name.lower():
        p.requires_grad_(False)

optimizer_p1 = torch.optim.AdamW(
    [p for p in decoder.parameters() if p.requires_grad], lr=LR_DECODER
)
n_batches = len(train_loader)

for epoch in tqdm(range(PHASE1_EPOCHS), desc='Phase 1'):
    decoder.train()
    for step, (imgs, masks, days, months, years) in enumerate(train_loader):
        imgs  = imgs.to(DEVICE)
        masks = masks.to(DEVICE)
        # Extract tokens live
        with torch.no_grad():
            tokens = extract_spatial_tokens(
                imgs.cpu().numpy(), days[0].item(), months[0].item(), years[0].item(),
                olmo_model, PATCH_SIZE, INPUT_SIZE, DEVICE,
            )
        tokens_t = torch.from_numpy(tokens).to(DEVICE)
        adjust_learning_rate(optimizer_p1, epoch, step, n_batches,
                              PHASE1_EPOCHS, LR_DECODER)
        optimizer_p1.zero_grad()
        logits = decoder(tokens_t.unsqueeze(0))
        loss   = loss_fn_live(logits, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(decoder.parameters(), 1.0)
        optimizer_p1.step()

torch.save({'model_state_dict': decoder.state_dict(), 'phase': 1},
           os.path.join(OUTPUT_DIR, 'checkpoints', 'phase1_end.pt'))
print('Phase 1 complete.')

## 9. Phase 2 — Decoder + LoRA Jointly

In [ ]:
# Unfreeze LoRA
for name, p in olmo_model.named_parameters():
    if 'lora' in name.lower():
        p.requires_grad_(True)

lora_params    = [p for n, p in olmo_model.named_parameters() if 'lora' in n.lower()]
decoder_params = [p for p in decoder.parameters() if p.requires_grad]

optimizer_p2 = torch.optim.AdamW([
    {'params': decoder_params, 'lr': LR_DECODER},
    {'params': lora_params,    'lr': LR_LORA},
])

best_val_f1, best_state = -1.0, None

for epoch in tqdm(range(PHASE2_EPOCHS), desc='Phase 2'):
    decoder.train()
    olmo_model.train()

    for step, (imgs, masks, days, months, years) in enumerate(train_loader):
        imgs  = imgs.to(DEVICE)
        masks = masks.to(DEVICE)
        tokens = extract_spatial_tokens(
            imgs.cpu().numpy(), days[0].item(), months[0].item(), years[0].item(),
            olmo_model, PATCH_SIZE, INPUT_SIZE, DEVICE,
        )
        tokens_t = torch.from_numpy(tokens).to(DEVICE)
        adjust_learning_rate(optimizer_p2, epoch, step, n_batches,
                              PHASE2_EPOCHS, LR_DECODER)
        optimizer_p2.zero_grad()
        logits = decoder(tokens_t.unsqueeze(0))
        loss   = loss_fn_live(logits, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(decoder.parameters()) + lora_params, 1.0)
        optimizer_p2.step()

    # Validate every 5 epochs
    if (epoch + 1) % 5 == 0 or epoch == PHASE2_EPOCHS - 1:
        decoder.eval(); olmo_model.eval()
        val_f1s = []
        with torch.no_grad():
            for imgs_v, masks_v, days_v, months_v, years_v in valid_loader:
                imgs_v = imgs_v.to(DEVICE); masks_v = masks_v.to(DEVICE)
                toks_v = extract_spatial_tokens(
                    imgs_v.cpu().numpy(), days_v[0].item(),
                    months_v[0].item(), years_v[0].item(),
                    olmo_model, PATCH_SIZE, INPUT_SIZE, DEVICE,
                )
                lgts_v = decoder(torch.from_numpy(toks_v).to(DEVICE).unsqueeze(0))
                val_f1s.append(compute_f1_argmax(lgts_v, masks_v))
        val_f1 = float(sum(val_f1s) / len(val_f1s))
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = {k: v.cpu().clone() for k, v in decoder.state_dict().items()}
            torch.save({'model_state_dict': best_state, 'val_f1': best_val_f1},
                       os.path.join(OUTPUT_DIR, 'checkpoints', 'best_lora.pt'))
        tqdm.write(f'  ep {PHASE1_EPOCHS + epoch + 1}: val_f1={val_f1:.4f}  '
                   f'(best={best_val_f1:.4f})')

print(f'Phase 2 complete.  Best val F1: {best_val_f1:.4f}')